In [1]:
# Environment & File Pre-checks

from pathlib import Path

ROOT = Path.cwd()
CHECKS = [
    (".env", "NEEDED", "Local environment configuration with storage directory paths"),
    (".env.example", "NEEDED", "Template environment file committed to repository"),
]

print(f"Checking execution directory: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<20}  {note}")

if missing:
    print(f"\n[ERROR] {missing} required file(s) missing from {ROOT}")
else:
    print("\nAll required environment configuration files are present.")

Checking execution directory: d:\NYU Bootcamp\bootcamp_Ziyu_Li\homework\homework05

  [OK ]  NEEDED    .env                  Local environment configuration with storage directory paths
  [OK ]  NEEDED    .env.example          Template environment file committed to repository

All required environment configuration files are present.


In [2]:
# Directory Setup via Environment Variables

import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

# Load environment configuration from .env
load_dotenv(override=True)

# Resolve target directories dynamically from environment variables
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))

# Automatically create directories if they do not exist
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

print('RAW Storage Directory  ->', RAW.resolve())
print('PROC Storage Directory ->', PROC.resolve())

RAW Storage Directory  -> D:\NYU Bootcamp\bootcamp_Ziyu_Li\homework\homework05\data\raw
PROC Storage Directory -> D:\NYU Bootcamp\bootcamp_Ziyu_Li\homework\homework05\data\processed


In [3]:
# Generate Sample Dataset

import numpy as np

# Create reproducible synthetic market data series
np.random.seed(42)
dates = pd.date_range('2026-01-01', periods=20, freq='D')
df = pd.DataFrame({
    'date': dates,
    'ticker': ['AAPL'] * 20,
    'price': 150 + np.random.randn(20).cumsum()
})

print("Sample DataFrame Preview:")
df.head()

Sample DataFrame Preview:


,date,ticker,price
0,2026-01-01,AAPL,150.496714
1,2026-01-02,AAPL,150.358450
2,2026-01-03,AAPL,151.006138
3,2026-01-04,AAPL,152.529168
4,2026-01-05,AAPL,152.295015


In [4]:
# ==========================================
# Task 1 — Save to Raw (CSV) & Processed (Parquet)
# ==========================================
def ts():
    """Return a formatted timestamp string for filename versioning."""
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# Save raw dataset in CSV format
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV to:", csv_path)

# Save processed dataset in Parquet format with defensive engine handling
pq_path = PROC / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path, index=False)
    print("Saved Parquet to:", pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    print('Error detail:', e)
    pq_path = None

Saved CSV to: data\raw\sample_20260901-150347.csv
Parquet engine not available. Install pyarrow or fastparquet to complete this step.
Error detail: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.


In [5]:
# ==========================================
# Task 2 — Reload & Validate Datasets
# ==========================================
def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame) -> dict:
    """Validate DataFrame integrity across shape and essential column dtypes."""
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']) if 'date' in reloaded.columns else False,
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']) if 'price' in reloaded.columns else False,
        'row_count': len(reloaded)
    }
    return checks

# 1. Reload and validate CSV
df_csv = pd.read_csv(csv_path, parse_dates=['date'])
csv_validation = validate_loaded(df, df_csv)
print("CSV Data Validation Results:", csv_validation)

# 2. Reload and validate Parquet
if pq_path and pq_path.exists():
    try:
        df_pq = pd.read_parquet(pq_path)
        pq_validation = validate_loaded(df, df_pq)
        print("Parquet Data Validation Results:", pq_validation)
    except Exception as e:
        print('Parquet reload failed:', e)

CSV Data Validation Results: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True, 'row_count': 20}


In [6]:
# ==========================================
# Task 3 — Abstract I/O Utilities
# ==========================================
import typing as t

def detect_format(path: t.Union[str, pathlib.Path]) -> str:
    """Detect file format based on suffix."""
    s = str(path).lower()
    if s.endswith('.csv'):
        return 'csv'
    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'):
        return 'parquet'
    raise ValueError(f'Unsupported file format suffix: {s}')

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]) -> pathlib.Path:
    """Route DataFrame writing based on suffix and auto-create parent directories."""
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p, index=False)
        except Exception as e:
            raise RuntimeError('Parquet engine unavailable. Install pyarrow or fastparquet.') from e
    return p

def read_df(path: t.Union[str, pathlib.Path]) -> pd.DataFrame:
    """Route DataFrame reading based on suffix with automatic date parsing."""
    p = pathlib.Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Target dataset file does not exist: {p}")
        
    fmt = detect_format(p)
    if fmt == 'csv':
        # Check column header to parse 'date' as datetime if present
        header = pd.read_csv(p, nrows=0).columns
        return pd.read_csv(p, parse_dates=['date']) if 'date' in header else pd.read_csv(p)
    else:
        try:
            return pd.read_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine unavailable. Install pyarrow or fastparquet.') from e

# Execute Utility Demonstrations
p_csv_util = RAW / f"util_{ts()}.csv"
p_pq_util  = PROC / f"util_{ts()}.parquet"

write_df(df, p_csv_util)
df_reloaded_csv = read_df(p_csv_util)
print("\nUtility CSV Reload Success. Shape:", df_reloaded_csv.shape)

try:
    write_df(df, p_pq_util)
    df_reloaded_pq = read_df(p_pq_util)
    print("Utility Parquet Reload Success. Shape:", df_reloaded_pq.shape)
except RuntimeError as e:
    print('Skipping Parquet utility demo due to engine absence:', e)


Utility CSV Reload Success. Shape: (20, 3)
Skipping Parquet utility demo due to engine absence: Parquet engine unavailable. Install pyarrow or fastparquet.


git add .  
git commit -m "feat(hw05): complete data storage layer implementation"  
git push origin main  